<a href="https://colab.research.google.com/github/njones61/xslope/blob/main/notebooks/xslope_seep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XSLOPE - Seepage Analysis

This notebook illustrates how to use xslope to solve a seepage problem using a 2D finite element solution.

## Install xslope and import functions

In [ ]:
# Install the xslope package
# - Basic install (limit equilibrium only): pip install xslope
# - Full install (with FEM features): pip install xslope[fem]

%%capture
!apt-get update && apt-get install -y libgl1-mesa-glx libglu1-mesa # required by gmsh
!pip install xslope[fem]==0.1.20
!pip install gmsh

In [ ]:
# Import functions

import xslope as xslope

from xslope.fileio import load_slope_data, print_dictionary
from xslope.mesh import build_polygons, build_mesh_from_polygons, export_mesh_to_json, import_mesh_from_json
from xslope.plot import plot_inputs, plot_mesh, plot_polygons, plot_polygons_separately
from xslope.plot_seep import plot_seep_data, plot_seep_solution
from xslope.seep import build_seep_data, run_seepage_analysis, save_seep_data_to_json, export_seep_solution

from pathlib import Path
import zipfile
from google.colab import files

## Upload Excel template

In [ ]:
# Upload excel input template or zip archive for selected problem.
# Zip archive can include mesh file from previous analysis

upload = files.upload()
file_name = list(upload.keys())[0]

# See if uploaded file is a zip archive. If so, unzip it
if file_name.endswith('.zip'):
  import zipfile
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
    # Extract all files
    zip_ref.extractall()
    extracted_files = zip_ref.namelist()

    # Find the excel file in the extracted list
    excel_file_found = False
    for f in extracted_files:
      if f.endswith('.xlsx'):
        file_name = f
        excel_file_found = True
        print(f"Found Excel file: {file_name}")
        break

    if not excel_file_found:
        # If no excel file is found at all, print an error and set file_name to None
        print("Error: No .xlsx file found in the uploaded archive. Please ensure your zip file contains an .xlsx file.")
        file_name = None

## Load slope data

In [ ]:
slope_data = load_slope_data(file_name)

plot_inputs(slope_data, mode='seep', save_png=False)

In [ ]:
# @title Select meshing options {"run":"auto"}
elem_type = "tri3" # @param ["tri3","tri6","quad4","quad8","quad9"]
elem_size = 4 # @param {"type":"number"}
auto_size = True # @param {"type":"boolean"}
# @markdown **Note:** If autosize is selected, elem_size will be ignored
re_mesh = False # @param {"type":"boolean"}

## Build mesh if necessary and then build and plot seep data

In [ ]:
# Use existing mesh from slope_data if available, otherwise build a new one
if slope_data.get("mesh") is not None and not re_mesh:
    print("Using existing mesh file.")
    mesh = slope_data["mesh"]
else:
    print("No existing mesh found in slope_data, building new mesh from profile line data.")
    polygons = build_polygons(slope_data)
    if auto_size:
        x_range = [min(x for x, _ in slope_data['ground_surface'].coords), max(x for x, _ in slope_data['ground_surface'].coords)]
        elem_size = (x_range[1] - x_range[0]) / 100
        print(f"Element size: {elem_size}")
    mesh = build_mesh_from_polygons(polygons, elem_size, elem_type)
    mesh_file = f"{Path(file_name).stem}_mesh.json"
    export_mesh_to_json(mesh, mesh_file)

seep_data = build_seep_data(mesh, slope_data)
has_seepage_bc2 = slope_data.get("has_seepage_bc2")

plot_seep_data(seep_data, show_nodes=True, show_bc=True, label_elements=False, label_nodes=False)

## Run seepage analysis

In [ ]:
solution = run_seepage_analysis(seep_data, tol=0.0001)

## Plot Results

In [ ]:
# @title Select plotting options {"run":"auto"}
variable = "head" # @param ["head","u","v_mag","i_mag"]
flowlines = True # @param {"type":"boolean"}
levels = 20 # @param {"type":"integer"}

base_mat = 2 # @param {"type":"integer"}
fill_contours = False # @param {"type":"boolean"}
alpha = 0.4 # @param {"type":"number"}
phreatic = True # @param {"type":"boolean"}

vectors = False # @param {"type":"boolean"}
vector_scale = 0.1 # @param {"type":"number"}

plot_mesh = False # @param {"type":"boolean"}

In [ ]:
plot_seep_solution(seep_data, solution, variable=variable, flowlines=flowlines, levels=levels, base_mat=base_mat, fill_contours=fill_contours,
                   alpha=alpha, phreatic=phreatic, vectors=vectors, vector_scale=vector_scale, mesh=plot_mesh)

## Process second set of boundary conditions if they exist

In [ ]:
# Check for a second set of seepage boundary conditions
if has_seepage_bc2:
    print("\nSecond set of seepage boundary conditions found. Running second analysis...")
    seep_data2 = build_seep_data(mesh, slope_data, seep_bc=2)
    plot_seep_data(seep_data2, figsize=(12, 6), show_nodes=True, show_bc=True, label_elements=False, label_nodes=False)
    solution2 = run_seepage_analysis(seep_data2, tol=1e-4)
    plot_seep_solution(seep_data2, solution2, variable=variable, flowlines=flowlines, levels=levels, base_mat=base_mat, fill_contours=fill_contours,
                   alpha=alpha, phreatic=phreatic, vectors=vectors, vector_scale=vector_scale, mesh=plot_mesh)
else:
  print("No second set of seepage boundary conditions found.")

## Save solution to CSV and download package

In [ ]:
# Save seep solution to CSV
input_path = Path(file_name)
seep_file_name = f"{input_path.stem}_seep.csv"
export_seep_solution(seep_data, solution, seep_file_name)
if has_seepage_bc2:
  seep_file_name2 = f"{input_path.stem}_seep2.csv"
  export_seep_solution(seep_data2, solution2, seep_file_name2)

# Identify the mesh file
mesh_file_name = f"{input_path.stem}_mesh.json"

# Create a zip file
zip_file_name = f"{input_path.stem}_results.zip"
with zipfile.ZipFile(zip_file_name, 'w') as zipf:
    zipf.write(file_name) # Original Excel file
    zipf.write(mesh_file_name) # Mesh JSON file
    zipf.write(seep_file_name) # Seep solution CSV file
    if has_seepage_bc2:
      zipf.write(seep_file_name2)

# Download the zip file
files.download(zip_file_name)